In [1]:
import os
from typing import Literal, Optional

import arxiv
import requests
from dotenv import load_dotenv
from pydantic import BaseModel

load_dotenv()

for key in ["GROQ_API_KEY", "OPENALEX_API_KEY"]:
    assert os.environ.get(key), f"Missing {key} in .env"

print("Environment loaded ✅")

Environment loaded ✅


In [2]:
class Paper(BaseModel):
    title: str
    authors: list[str]
    year: int | None = None
    abstract: str | None = None
    url: str | None = None
    pdf_url: str | None = None
    citation_count: int | None = None
    source: Literal["arxiv", "openalex", "semantic_scholar"]

In [3]:
import time

_last_ss_call = 0.0
SEMANTIC_SCHOLAR_BASE = "https://api.semanticscholar.org/graph/v1/paper/search"


def search_semantic_scholar(query: str, max_results: int = 5) -> list[Paper]:
    global _last_ss_call
    elapsed = time.time() - _last_ss_call
    if elapsed < 1.0:
        time.sleep(1.0 - elapsed)
    _last_ss_call = time.time()

    headers = {"x-api-key": os.environ["SEMANTIC_SCHOLAR_API_KEY"]}
    params = {
        "query": query,
        "limit": max_results,
        "fields": "title,year,abstract,authors,citationCount,openAccessPdf",
    }
    try:
        resp = requests.get(
            SEMANTIC_SCHOLAR_BASE, params=params, headers=headers, timeout=15
        )
        resp.raise_for_status()
        data = resp.json().get("data", [])
    except Exception as e:
        print(f"⚠️ Semantic Scholar search failed ({type(e).__name__}); skipping")
        return []

    return [
        Paper(
            title=p.get("title") or "Untitled",
            authors=[a["name"] for a in p.get("authors", [])],
            year=p.get("year"),
            abstract=p.get("abstract"),
            url=f"https://www.semanticscholar.org/paper/{p['paperId']}"
            if p.get("paperId")
            else None,
            pdf_url=(p.get("openAccessPdf") or {}).get("url"),
            citation_count=p.get("citationCount"),
            source="semantic_scholar",
        )
        for p in data
    ]

In [4]:
def search_arxiv(query: str, max_results: int = 5) -> list[Paper]:
    client = arxiv.Client(page_size=max_results, delay_seconds=3.0, num_retries=2)
    search = arxiv.Search(
        query=query, max_results=max_results, sort_by=arxiv.SortCriterion.Relevance
    )
    try:
        return [
            Paper(
                title=r.title,
                authors=[a.name for a in r.authors],
                year=r.published.year,
                abstract=r.summary.replace("\n", " "),
                url=r.entry_id,
                pdf_url=r.pdf_url,
                source="arxiv",
            )
            for r in client.results(search)
        ]
    except Exception as e:
        print(
            f"⚠️ arXiv search failed ({type(e).__name__}); continuing with OpenAlex only"
        )
        return []

In [5]:
OPENALEX_BASE = "https://api.openalex.org/works"


def _reconstruct_abstract(inverted_index: dict | None) -> str | None:
    if not inverted_index:
        return None
    positions = {}
    for word, idxs in inverted_index.items():
        for idx in idxs:
            positions[idx] = word
    return " ".join(positions[i] for i in sorted(positions))


def search_openalex(query: str, max_results: int = 5) -> list[Paper]:
    params = {
        "search": query,
        "per_page": max_results,
        "select": "title,authorships,publication_year,abstract_inverted_index,id,open_access,cited_by_count",
        "api_key": os.environ["OPENALEX_API_KEY"],
    }
    resp = requests.get(OPENALEX_BASE, params=params, timeout=15)
    resp.raise_for_status()

    results = []
    for w in resp.json()["results"]:
        results.append(
            Paper(
                title=w.get("title") or "Untitled",
                authors=[a["author"]["display_name"] for a in w.get("authorships", [])],
                year=w.get("publication_year"),
                abstract=_reconstruct_abstract(w.get("abstract_inverted_index")),
                url=w.get("id"),
                pdf_url=(w.get("open_access") or {}).get("oa_url"),
                citation_count=w.get("cited_by_count"),
                source="openalex",
            )
        )
    return results

In [6]:
def research_search(query: str, max_results: int = 5) -> list[Paper]:
    papers = (
        search_arxiv(query, max_results)
        + search_openalex(query, max_results)
        + search_semantic_scholar(query, max_results)
    )
    seen, deduped = set(), []
    for p in papers:
        key = p.title.strip().lower()
        if key not in seen:
            seen.add(key)
            deduped.append(p)
    return deduped

In [7]:
from langchain_core.tools import tool


@tool
def research_papers(query: str, max_results: int = 5) -> list[dict]:
    """Search arXiv, OpenAlex and Semantic Scholar for academic papers relevant to a learning topic.
    Use this when the user wants to learn about or research a concept and source
    material is needed to ground an explanation."""
    return [p.model_dump() for p in research_search(query, max_results)]

### RAG

In [8]:
import psycopg2
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pgvector.psycopg2 import register_vector
from psycopg2.extras import Json
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

conn = psycopg2.connect(os.environ["SUPABASE_DB_URL"])
conn.autocommit = True
register_vector(conn)
print("Connected to Supabase ✅")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Connected to Supabase ✅


In [9]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)


def chunk_paper(paper: Paper) -> list[str]:
    text = paper.abstract or ""
    return splitter.split_text(text) if text else []

In [10]:
def ingest_papers(papers: list[Paper]) -> int:
    inserted = 0
    with conn.cursor() as cur:
        for paper in papers:
            chunks = chunk_paper(paper)
            if not chunks:
                continue
            embeddings = embedder.encode(chunks)
            for chunk_text, embedding in zip(chunks, embeddings):
                cur.execute(
                    """insert into paper_chunks
                    (paper_title, paper_url, source, chunk_text, embedding, metadata)
                    values (%s, %s, %s, %s, %s::vector, %s)""",
                    (
                        paper.title,
                        paper.url,
                        paper.source,
                        chunk_text,
                        embedding,
                        Json(
                            {"year": paper.year, "citation_count": paper.citation_count}
                        ),
                    ),
                )
                inserted += 1
    return inserted

In [11]:
def retrieve_context(query: str, top_k: int = 5) -> list[dict]:
    query_embedding = embedder.encode([query])[0]
    with conn.cursor() as cur:
        cur.execute(
            "select * from match_paper_chunks(%s::vector, %s)", (query_embedding, top_k)
        )
        cols = [d[0] for d in cur.description]
        return [dict(zip(cols, row)) for row in cur.fetchall()]

In [12]:
papers = research_search("retrieval augmented generation", max_results=5)
print(f"Ingested {ingest_papers(papers)} chunks")

for r in retrieve_context("How does RAG reduce hallucination?", top_k=3):
    print(f"[{r['similarity']:.3f}] {r['paper_title']} — {r['chunk_text'][:100]}...")

⚠️ arXiv search failed (HTTPError); continuing with OpenAlex only
⚠️ Semantic Scholar search failed (HTTPError); skipping
Ingested 12 chunks
[0.503] Retrieval-Augmented Generation for Large Language Models: A Survey — and the augmentation techniques. The paper highlights the state-of-the-art technologies embedded in ...
[0.455] Benchmarking Large Language Models in Retrieval-Augmented Generation — that there is still a considerable journey ahead to effectively apply RAG to LLMs....
[0.407] Benchmarking Retrieval-Augmented Generation for Medicine — improves the accuracy of six different LLMs by up to 18% over chain-of-thought prompting, elevating ...


### Extractor

In [13]:
from langchain_groq import ChatGroq
from mem0 import MemoryClient

USER_ID = "prashant"

llm = ChatGroq(
    model="openai/gpt-oss-120b", temperature=0.3, api_key=os.environ["GROQ_API_KEY"]
)
mem0_client = MemoryClient(api_key=os.environ["MEM0_API_KEY"])

In [14]:
def get_learner_context(topic: str, limit: int = 5) -> str:
    response = mem0_client.search(topic, filters={"user_id": USER_ID}, limit=limit)
    memories = response.get("results", []) if isinstance(response, dict) else response
    if not memories:
        return "No prior context on this topic yet — first time it's coming up."
    return "\n".join(f"- {m['memory']}" for m in memories)


def store_correction(topic: str, user_reaction: str):
    messages = [
        {"role": "user", "content": f"Regarding the topic '{topic}': {user_reaction}"}
    ]
    mem0_client.add(messages, user_id=USER_ID, metadata={"topic": topic})

In [15]:
EXPLAINER_SYSTEM_PROMPT = """You are a personal tutor explaining a concept to one specific learner.
Use ONLY the retrieved source material to ground your explanation — never invent facts.
Adapt tone and depth to what you already know about this learner's preferences.
If there's no prior context, default to a clear, moderately technical explanation with a concrete example."""


def explain_topic(topic: str) -> dict:
    sources = retrieve_context(topic, top_k=5)
    source_text = "\n\n".join(
        f"[{s['paper_title']}]: {s['chunk_text']}" for s in sources
    )
    learner_context = get_learner_context(topic)

    prompt = f"""Topic: {topic}

Retrieved source material:
{source_text}

What I know about this learner:
{learner_context}

Explain this topic to the learner now."""

    response = llm.invoke(
        [
            {"role": "system", "content": EXPLAINER_SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ]
    )
    return {"explanation": response.content, "sources": sources}

In [16]:
CRITIQUE_SYSTEM_PROMPT = """You are a fact-checker. Compare the explanation against the source material.
Flag any claim NOT supported by the sources, or that contradicts them.
Respond with exactly "PASS" if fully grounded.
Otherwise respond with "REVISE: <specific issue to fix>"."""

def critique_explanation(explanation: str, sources: list[dict]) -> str:
    source_text = "\n\n".join(s["chunk_text"] for s in sources)
    response = llm.invoke([
        {"role": "system", "content": CRITIQUE_SYSTEM_PROMPT},
        {"role": "user", "content": f"Source material:\n{source_text}\n\nExplanation to check:\n{explanation}"},
    ])
    return response.content.strip()

In [17]:
def explain_with_self_correction(topic: str, max_retries: int = 2) -> dict:
    result = explain_topic(topic)
    for attempt in range(max_retries):
        verdict = critique_explanation(result["explanation"], result["sources"])
        if verdict.upper().startswith("PASS"):
            break
        print(f"🔁 Critique flagged an issue (attempt {attempt + 1}): {verdict}")
        response = llm.invoke([
            {"role": "system", "content": EXPLAINER_SYSTEM_PROMPT},
            {"role": "user", "content": f"Your previous explanation had an issue: {verdict}\nRevise it, still grounded only in the source material."},
        ])
        result["explanation"] = response.content
    return result

In [18]:
def learn_about(topic: str):
    result = explain_with_self_correction(topic)
    print(result["explanation"])
    print("\n---")
    reaction = input("Your reaction (e.g. 'got it', 'too basic', 'use an analogy', 'I already know this'): ")
    store_correction(topic, reaction)
    return reaction

In [19]:
learn_about("retrieval augmented generation")

**Retrieval‑Augmented Generation (RAG) – an analogy**

Imagine you are a student writing a research essay.  

1. **Your own knowledge** – The ideas, writing style, and reasoning you already have in your head are like the *large language model (LLM)* itself. It can produce fluent text, but sometimes it “hallucinates” facts that aren’t true, just as a student might guess a citation when they can’t remember the source.  

2. **The library** – The university library (or an online database) contains millions of verified books, articles, and data. This is the *external knowledge base* that RAG can query.  

3. **The librarian** – When you need a specific piece of information, you ask the librarian to fetch the most relevant books or passages. In RAG, this role is played by a *retriever* that searches the external database for documents that match the current prompt.  

4. **Writing the essay** – After the librarian hands you the relevant excerpts, you read them, synthesize the facts, and wea

'got it'